In [1]:
import pandas as pd

# Load datasets
rtt_df = pd.read_csv("cleaned_rtt_analysis.csv")
disorder_df = pd.read_csv("disorder_type_distribution.csv")

drug_df = pd.read_csv("drug_summary.csv")
icb_df = pd.read_csv("icb_category_summary.csv")
monthly_df = pd.read_csv("monthly_trend_summary.csv")
regional_df = pd.read_csv("regional_summary.csv")

# Print columns
print("RTT Columns:")
print(rtt_df.columns)

print("\nDisorder Columns:")
print(disorder_df.columns)

print("\nDrug Columns:")
print(drug_df.columns)

print("\nICB Columns:")
print(icb_df.columns)

print("\nMonthly Columns:")
print(monthly_df.columns)

print("\nRegional Columns:")
print(regional_df.columns)

RTT Columns:
Index(['Region', 'Mean_RTT', 'Median_RTT'], dtype='object')

Disorder Columns:
Index(['MEASURE_NAME', 'count'], dtype='object')

Drug Columns:
Index(['BNF_CHEMICAL_SUBSTANCE', 'TOTAL_QUANTITY'], dtype='object')

ICB Columns:
Index(['ICB_NAME', 'Antidepressant', 'Anxiolytic'], dtype='object')

Monthly Columns:
Index(['YEAR_MONTH', 'TOTAL_QUANTITY'], dtype='object')

Regional Columns:
Index(['REGIONAL_OFFICE_NAME', 'TOTAL_QUANTITY'], dtype='object')


In [2]:
from dash import Dash, dcc, html
import plotly.express as px
import pandas as pd

# =========================
# LOAD DATASETS
# =========================

rtt_df = pd.read_csv("cleaned_rtt_analysis.csv")
disorder_df = pd.read_csv("disorder_type_distribution.csv")

drug_df = pd.read_csv("drug_summary.csv")
icb_df = pd.read_csv("icb_category_summary.csv")
monthly_df = pd.read_csv("monthly_trend_summary.csv")
regional_df = pd.read_csv("regional_summary.csv")

# =========================
# CREATE FIGURES
# =========================

# Mean RTT Chart
fig_mean_rtt = px.bar(
    rtt_df,
    x='Region',
    y='Mean_RTT',
    title='Mean RTT by Region'
)

# Median RTT Chart
fig_median_rtt = px.bar(
    rtt_df,
    x='Region',
    y='Median_RTT',
    title='Median RTT by Region'
)

# Disorder Distribution
fig_disorder = px.pie(
    disorder_df,
    names='MEASURE_NAME',
    values='count',
    title='Disorder Type Distribution'
)

# Monthly Trend
fig_monthly = px.line(
    monthly_df,
    x='YEAR_MONTH',
    y='TOTAL_QUANTITY',
    title='Monthly Prescribing Trend'
)

# Regional Analysis
fig_regional = px.bar(
    regional_df,
    x='REGIONAL_OFFICE_NAME',
    y='TOTAL_QUANTITY',
    title='Regional Prescribing Analysis'
)

# Drug Comparison
fig_drug = px.bar(
    drug_df,
    x='BNF_CHEMICAL_SUBSTANCE',
    y='TOTAL_QUANTITY',
    title='Drug Quantity Comparison'
)

# ICB Comparison
fig_icb = px.bar(
    icb_df,
    x='ICB_NAME',
    y=['Antidepressant', 'Anxiolytic'],
    barmode='group',
    title='ICB-wise Antidepressant vs Anxiolytic'
)

# =========================
# CREATE DASH APP
# =========================

app = Dash(__name__)

app.layout = html.Div([

    html.H1(
        "NHS Mental Health Analytics Dashboard",
        style={'textAlign': 'center'}
    ),

    html.H2("IAPT Analysis"),

    dcc.Graph(figure=fig_mean_rtt),

    dcc.Graph(figure=fig_median_rtt),

    dcc.Graph(figure=fig_disorder),

    html.H2("Prescribing Analysis"),

    dcc.Graph(figure=fig_monthly),

    dcc.Graph(figure=fig_regional),

    dcc.Graph(figure=fig_drug),

    dcc.Graph(figure=fig_icb)

])

# =========================
# RUN APP
# =========================

app.run(debug=True)

Dash is running on http://127.0.0.1:8050/



INFO:dash.dash:Dash is running on http://127.0.0.1:8050/



 * Serving Flask app '__main__'
 * Debug mode: on


In [4]:
!pip install pyngrok

In [63]:
from pyngrok import ngrok

!ngrok config add-authtoken 3COFKiRIiqt9YznkUtRLMNEbsXu_74J6dHmVrRkxpJTpLRMuX

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [39]:
from pyngrok import ngrok

public_url = ngrok.connect(8050)

print(public_url)

NgrokTunnel: "https://simplify-dragonfly-fidgeting.ngrok-free.dev" -> "http://localhost:8050"


In [ ]:
app.run(port=8050)

Dash is running on http://127.0.0.1:8050/



INFO:dash.dash:Dash is running on http://127.0.0.1:8050/



 * Serving Flask app '__main__'
 * Debug mode: off


In [38]:
from dash import Dash, dcc, html, Input, Output
import plotly.express as px
import pandas as pd

# ==========================================
# LOAD DATASETS
# ==========================================

rtt_df = pd.read_csv("cleaned_rtt_analysis.csv")
disorder_df = pd.read_csv("disorder_type_distribution.csv")

drug_df = pd.read_csv("drug_summary.csv")
icb_df = pd.read_csv("icb_category_summary.csv")
monthly_df = pd.read_csv("monthly_trend_summary.csv")
regional_df = pd.read_csv("regional_summary.csv")

# ==========================================
# KPI VALUES
# ==========================================

total_prescriptions = int(regional_df['TOTAL_QUANTITY'].sum())
total_regions = regional_df['REGIONAL_OFFICE_NAME'].nunique()
avg_mean_rtt = round(rtt_df['Mean_RTT'].mean(), 2)

# ==========================================
# CREATE DASH APP
# ==========================================

app = Dash(__name__)

# ==========================================
# APP LAYOUT
# ==========================================

app.layout = html.Div([

    # ======================================
    # TITLE
    # ======================================

    html.H1(
        "NHS Mental Health Analytics Dashboard",
        style={
            'textAlign': 'center',
            'marginBottom': '35px',
            'marginTop': '20px',
            'fontSize': '42px',
            'fontWeight': 'bold'
        }
    ),

    # ======================================
    # DROPDOWNS
    # ======================================

    html.Div([

        # REGION DROPDOWN

        html.Div([

            html.Label(
                "Select Region",
                style={
                    'fontWeight': 'bold',
                    'fontSize': '20px'
                }
            ),

            dcc.Dropdown(
                id='region-dropdown',

                options=[
                    {
                        'label': region,
                        'value': region
                    }
                    for region in regional_df[
                        'REGIONAL_OFFICE_NAME'
                    ].unique()
                ],

                value=regional_df[
                    'REGIONAL_OFFICE_NAME'
                ].unique()[0],

                style={
                    'fontSize': '18px'
                }
            )

        ], style={
            'width': '49%',
            'display': 'inline-block'
        }),

        # CATEGORY DROPDOWN

        html.Div([

            html.Label(
                "Select Category",
                style={
                    'fontWeight': 'bold',
                    'fontSize': '20px'
                }
            ),

            dcc.Dropdown(
                id='category-dropdown',

                options=[
                    {
                        'label': 'Antidepressant',
                        'value': 'Antidepressant'
                    },
                    {
                        'label': 'Anxiolytic',
                        'value': 'Anxiolytic'
                    }
                ],

                value='Antidepressant',

                style={
                    'fontSize': '18px'
                }
            )

        ], style={
            'width': '49%',
            'display': 'inline-block',
            'float': 'right'
        })

    ], style={
        'marginBottom': '40px'
    }),

    # ======================================
    # KPI CARDS
    # ======================================

    html.Div([

        html.Div([

            html.H3(
                "Total Prescriptions",
                style={'fontSize': '24px'}
            ),

            html.H1(
                f"{total_prescriptions:,}",
                style={'fontSize': '38px'}
            )

        ], style={
            'width': '31%',
            'padding': '25px',
            'borderRadius': '12px',
            'boxShadow': '0px 0px 10px lightgray',
            'textAlign': 'center',
            'backgroundColor': '#f8f9fa'
        }),

        html.Div([

            html.H3(
                "Total Regions",
                style={'fontSize': '24px'}
            ),

            html.H1(
                f"{total_regions}",
                style={'fontSize': '38px'}
            )

        ], style={
            'width': '31%',
            'padding': '25px',
            'borderRadius': '12px',
            'boxShadow': '0px 0px 10px lightgray',
            'textAlign': 'center',
            'backgroundColor': '#f8f9fa'
        }),

        html.Div([

            html.H3(
                "Average Mean RTT",
                style={'fontSize': '24px'}
            ),

            html.H1(
                f"{avg_mean_rtt}",
                style={'fontSize': '38px'}
            )

        ], style={
            'width': '31%',
            'padding': '25px',
            'borderRadius': '12px',
            'boxShadow': '0px 0px 10px lightgray',
            'textAlign': 'center',
            'backgroundColor': '#f8f9fa'
        })

    ], style={
        'display': 'flex',
        'justifyContent': 'space-between',
        'marginBottom': '50px'
    }),

    # ======================================
    # IAPT ANALYSIS
    # ======================================

    html.H2(
        "IAPT Analysis",
        style={
            'fontSize': '34px',
            'marginBottom': '25px'
        }
    ),

    # ======================================
    # MEAN RTT GRAPH
    # ======================================

    html.Div([
        dcc.Graph(id='mean-rtt-chart')
    ], style={
        'width': '90%',
        'margin': 'auto',
        'marginBottom': '35px'
    }),

    # ======================================
    # MEDIAN RTT GRAPH
    # ======================================

    html.Div([
        dcc.Graph(id='median-rtt-chart')
    ], style={
        'width': '90%',
        'margin': 'auto',
        'marginBottom': '35px'
    }),

    # ======================================
    # DISORDER GRAPH
    # ======================================

    html.Div([
        dcc.Graph(id='disorder-chart')
    ], style={
        'width': '75%',
        'margin': 'auto',
        'marginBottom': '35px'
    }),

    # ======================================
    # PRESCRIBING ANALYSIS
    # ======================================

    html.H2(
        "Prescribing Analysis",
        style={
            'fontSize': '34px',
            'marginTop': '40px',
            'marginBottom': '25px'
        }
    ),

    # ======================================
    # MONTHLY GRAPH
    # ======================================

    html.Div([
        dcc.Graph(id='monthly-chart')
    ], style={
        'width': '90%',
        'margin': 'auto',
        'marginBottom': '35px'
    }),

    # ======================================
    # REGIONAL GRAPH
    # ======================================

    html.Div([
        dcc.Graph(id='regional-chart')
    ], style={
        'width': '90%',
        'margin': 'auto',
        'marginBottom': '35px'
    }),

    # ======================================
    # DRUG GRAPH
    # ======================================

    html.Div([
        dcc.Graph(id='drug-chart')
    ], style={
        'width': '90%',
        'margin': 'auto',
        'marginBottom': '35px'
    }),

    # ======================================
    # ICB GRAPH
    # ======================================

    html.Div([
        dcc.Graph(id='icb-chart')
    ], style={
        'width': '95%',
        'margin': 'auto',
        'marginBottom': '35px'
    }),

], style={
    'padding': '20px 30px',
    'width': '100%',
    'fontFamily': 'Arial',
    'backgroundColor': 'white'
})

# ==========================================
# CALLBACKS
# ==========================================

@app.callback(

    [
        Output('regional-chart', 'figure'),
        Output('mean-rtt-chart', 'figure'),
        Output('median-rtt-chart', 'figure'),
        Output('drug-chart', 'figure'),
        Output('icb-chart', 'figure'),
        Output('monthly-chart', 'figure'),
        Output('disorder-chart', 'figure')
    ],

    [
        Input('region-dropdown', 'value'),
        Input('category-dropdown', 'value')
    ]

)

def update_all_charts(selected_region, selected_category):

    # ======================================
    # REGIONAL FILTER
    # ======================================

    filtered_regional = regional_df.copy()

    filtered_regional['Color'] = filtered_regional[
        'REGIONAL_OFFICE_NAME'
    ].apply(
        lambda x: 'Selected'
        if x == selected_region
        else 'Other'
    )

    # ======================================
    # REGIONAL GRAPH
    # ======================================

    regional_fig = px.bar(
        filtered_regional,
        x='REGIONAL_OFFICE_NAME',
        y='TOTAL_QUANTITY',
        color='Color',
        title=f'{selected_category} Analysis by Region',
        color_discrete_map={
            'Selected': 'crimson',
            'Other': 'lightgray'
        }
    )

    regional_fig.update_layout(
        height=500,
        title_x=0.5,
        title_font=dict(size=26),

        xaxis=dict(
            title_font=dict(size=18),
            tickfont=dict(size=12)
        ),

        yaxis=dict(
            title_font=dict(size=18),
            tickfont=dict(size=14)
        )
    )

    # ======================================
    # RTT FILTER
    # ======================================

    matched_rtt = rtt_df.copy()

    matched_rtt['Color'] = matched_rtt[
        'Region'
    ].apply(
        lambda x: 'Selected'
        if selected_region.split()[0] in x.upper()
        else 'Other'
    )

    # ======================================
    # MEAN RTT GRAPH
    # ======================================

    mean_rtt_fig = px.bar(
        matched_rtt,
        x='Region',
        y='Mean_RTT',
        color='Color',
        title='Mean RTT Analysis by Region',
        color_discrete_map={
            'Selected': 'royalblue',
            'Other': 'lightgray'
        }
    )

    mean_rtt_fig.update_layout(
        height=500,
        title_x=0.5,
        title_font=dict(size=26),

        xaxis=dict(
            title_font=dict(size=18),
            tickfont=dict(size=11)
        ),

        yaxis=dict(
            title_font=dict(size=18),
            tickfont=dict(size=14)
        )
    )

    # ======================================
    # MEDIAN RTT GRAPH
    # ======================================

    median_rtt_fig = px.bar(
        matched_rtt,
        x='Region',
        y='Median_RTT',
        color='Color',
        title='Median RTT Analysis by Region',
        color_discrete_map={
            'Selected': 'purple',
            'Other': 'lightgray'
        }
    )

    median_rtt_fig.update_layout(
        height=500,
        title_x=0.5,
        title_font=dict(size=26),

        xaxis=dict(
            title_font=dict(size=18),
            tickfont=dict(size=11)
        ),

        yaxis=dict(
            title_font=dict(size=18),
            tickfont=dict(size=14)
        )
    )

    # ======================================
    # DISORDER GRAPH
    # ======================================

    disorder_fig = px.pie(
        disorder_df,
        names='MEASURE_NAME',
        values='count',
        title=f'Disorder Distribution - {selected_region}'
    )

    disorder_fig.update_layout(
        height=500,
        title_x=0.5,
        title_font=dict(size=26),

        legend=dict(
            font=dict(size=16)
        )
    )

    # ======================================
    # MONTHLY GRAPH
    # ======================================

    monthly_fig = px.line(
        monthly_df,
        x='YEAR_MONTH',
        y='TOTAL_QUANTITY',
        title=f'{selected_category} Monthly Trend',
        markers=True
    )

    monthly_fig.update_layout(
        height=500,
        title_x=0.5,
        title_font=dict(size=26),

        xaxis=dict(
            title_font=dict(size=18),
            tickfont=dict(size=12)
        ),

        yaxis=dict(
            title_font=dict(size=18),
            tickfont=dict(size=14)
        )
    )

    # ======================================
    # DRUG GRAPH
    # ======================================

    drug_fig = px.bar(
        drug_df,
        x='BNF_CHEMICAL_SUBSTANCE',
        y='TOTAL_QUANTITY',
        title=f'{selected_category} Drug Comparison',
        color='TOTAL_QUANTITY'
    )

    drug_fig.update_layout(
        height=500,
        title_x=0.5,
        title_font=dict(size=26),

        xaxis=dict(
            title_font=dict(size=18),
            tickfont=dict(size=11)
        ),

        yaxis=dict(
            title_font=dict(size=18),
            tickfont=dict(size=14)
        )
    )

    # ======================================
    # ICB GRAPH
    # ======================================

    icb_fig = px.bar(
        icb_df,
        x='ICB_NAME',
        y=[selected_category],
        title=f'ICB-wise {selected_category} Analysis'
    )

    icb_fig.update_layout(
        height=550,
        title_x=0.5,
        title_font=dict(size=26),

        xaxis=dict(
            title_font=dict(size=18),
            tickfont=dict(size=10)
        ),

        yaxis=dict(
            title_font=dict(size=18),
            tickfont=dict(size=14)
        )
    )

    return (
        regional_fig,
        mean_rtt_fig,
        median_rtt_fig,
        drug_fig,
        icb_fig,
        monthly_fig,
        disorder_fig
    )

